# 🏕️ CampingTN — Site Recommender & Equipment Classifier
## Notebook 2: Recommendation Engine + Equipment Need Predictor

This notebook builds:
1. **Camping Site Recommender** — collaborative filtering + content-based hybrid
2. **Equipment Need Classifier** — predicts which gear items are essential given conditions
3. **Regional Trend Analysis** — time-series analysis of 2002-2017 dataset

**Export:** `recommender.pkl`, `equipment_classifier.pkl`

In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn joblib scipy -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, MinMaxScaler, MultiLabelBinarizer
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, accuracy_score, hamming_loss
from scipy.sparse import csr_matrix

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)
print('✅ Libraries loaded')

## Part 1: Regional Trend Analysis (2002–2017 Dataset)

In [ ]:
# Official regional distribution data
regional_data = {
    'region': ['District-Tunis','Nord-Est','Nord-Ouest','Centre-Est','Centre-Ouest','Sud-Est','Sud-Ouest'],
    'centers_2017': [2, 3, 7, 5, 2, 3, 1],
    'capacity_buildings': [32, 125, 268, 270, 96, 64, 30],
    'capacity_tents': [80, 45, 220, 100, 120, 200, 200],
    'site_types': ['FOREST','COASTAL/FOREST','FOREST','COASTAL','FOREST/DESERT','COASTAL/DESERT','DESERT'],
}

evolution_data = {
    'year': [2002,2004,2006,2008,2010,2012,2014,2016,2017],
    'District-Tunis': [2,2,2,2,2,2,2,1,0],
    'Nord-Est': [4,4,4,4,4,3,3,3,0],
    'Nord-Ouest': [7,7,7,7,7,7,7,7,0],
    'Centre-Est': [4,4,4,4,6,7,7,5,1],
    'Centre-Ouest': [1,1,1,2,2,2,2,2,0],
    'Sud-Est': [4,4,4,4,4,4,4,3,0],
    'Sud-Ouest': [1,1,1,1,1,1,1,1,1],
}

df_regional = pd.DataFrame(regional_data)
df_evo = pd.DataFrame(evolution_data)

print('Regional Distribution 2017:')
print(df_regional.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Tunisia Camping Centers — Regional Analysis 2002-2017', fontsize=16, fontweight='bold')

colors = ['#2d6a4f','#52b788','#0077b6','#e9c46a','#f4a261','#e63946','#8b5cf6']
regions = df_regional['region'].tolist()

# Centers per region
bars = axes[0,0].bar(regions, df_regional['centers_2017'], color=colors)
axes[0,0].set_title('Number of Centers per Region (2017)', fontweight='bold')
axes[0,0].set_ylabel('Number of Centers')
axes[0,0].tick_params(axis='x', rotation=30)
for bar, val in zip(bars, df_regional['centers_2017']):
    axes[0,0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                   str(val), ha='center', va='bottom', fontweight='bold')

# Evolution over time
for i, region in enumerate(regions):
    total = [df_evo[region].iloc[y] for y in range(len(df_evo))]
    axes[0,1].plot(df_evo['year'], total, marker='o', label=region, color=colors[i], linewidth=2)
axes[0,1].set_title('Evolution of Centers per Region (2002-2017)', fontweight='bold')
axes[0,1].set_ylabel('Number of Centers')
axes[0,1].set_xlabel('Year')
axes[0,1].legend(fontsize=7, ncol=2)

# Capacity comparison
x = np.arange(len(regions))
w = 0.35
axes[1,0].bar(x - w/2, df_regional['capacity_buildings'], w, label='Buildings', color='#2d6a4f')
axes[1,0].bar(x + w/2, df_regional['capacity_tents'], w, label='Tents', color='#52b788')
axes[1,0].set_title('Capacity by Type per Region (2017)', fontweight='bold')
axes[1,0].set_ylabel('Capacity (persons)')
axes[1,0].set_xticks(x)
axes[1,0].set_xticklabels(regions, rotation=30, ha='right')
axes[1,0].legend()

# Site types donut chart
type_counts = {'Forest': 7, 'Coastal': 5, 'Desert': 3, 'Mixed': 7}
wedges, texts, autotexts = axes[1,1].pie(
    type_counts.values(), labels=type_counts.keys(),
    autopct='%1.1f%%', colors=['#2d6a4f','#0077b6','#e9c46a','#52b788'],
    pctdistance=0.8, startangle=90,
    wedgeprops=dict(width=0.5))
axes[1,1].set_title('Distribution by Site Nature (2017)', fontweight='bold')
centre_circle = plt.Circle((0,0), 0.3, fc='white')
axes[1,1].add_artist(centre_circle)

plt.tight_layout()
plt.savefig('regional_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Regional analysis saved')

## Part 2: Camping Site Recommender System

In [ ]:
# Build content-based feature matrix for camping centers
centers = pd.DataFrame({
    'id': range(13),
    'name': ['Centre El Hbibia','Centre Chat Mami','Centre Errimal','Centre Beni Mtir',
             'Centre Ain Bousaadia','Centre El Salloume','Centre El Douirat (Mahdia)',
             'Centre Erramela','Centre Ain Selsla','Centre El Cheaanbi',
             'Centre Marsa El Kssiba','Centre El Douirat (Tataouine)','Centre Douz'],
    'site_type': ['FOREST','COASTAL','COASTAL','FOREST','FOREST','COASTAL','COASTAL',
                  'COASTAL','FOREST','DESERT','COASTAL','DESERT','DESERT'],
    'region': ['District-Tunis','Nord-Est','Nord-Est','Nord-Ouest','Nord-Ouest',
               'Centre-Est','Centre-Est','Centre-Est','Centre-Ouest','Centre-Ouest',
               'Sud-Est','Sud-Est','Sud-Ouest'],
    'capacity': [80,75,95,100,132,130,80,80,130,86,264,0,230],
    'has_buildings': [0,1,1,1,1,1,1,1,1,1,1,0,1],
    'has_tents': [1,1,0,0,1,1,0,0,1,1,1,0,1],
    'rating': [3.8,4.2,4.0,4.5,3.9,4.6,4.3,3.7,4.1,4.4,4.7,4.8,4.9],
    'year_created': [1989,None,None,None,1989,1999,2011,1986,1963,2008,1987,1996,1990],
    'is_north': [1,1,1,1,1,0,0,0,0,0,0,0,0],
    'is_south': [0,0,0,0,0,0,0,0,0,0,1,1,1],
})

# One-hot encode site type and region
centers_encoded = pd.get_dummies(centers, columns=['site_type', 'region'])
numeric_cols = [c for c in centers_encoded.columns
                if c not in ['id', 'name', 'year_created']]
centers_encoded[numeric_cols] = centers_encoded[numeric_cols].fillna(0)

# Normalize
scaler = MinMaxScaler()
feature_cols = [c for c in numeric_cols]
centers_scaled = scaler.fit_transform(centers_encoded[feature_cols])

# KNN recommender
knn = NearestNeighbors(n_neighbors=5, metric='cosine', algorithm='brute')
knn.fit(centers_scaled)

print(f'Recommender built on {len(centers)} camping centers')
print(f'Feature matrix shape: {centers_scaled.shape}')

In [ ]:
def recommend_centers(preferences: dict, top_n: int = 5) -> pd.DataFrame:
    """
    Recommend camping centers based on user preferences.
    preferences: dict with keys like site_type, is_south, capacity_needed, etc.
    """
    # Build preference vector matching feature columns
    pref_vector = np.zeros(centers_scaled.shape[1])
    feature_list = feature_cols

    # Site type preference
    for i, col in enumerate(feature_list):
        if 'site_type_' + preferences.get('site_type', '') == col:
            pref_vector[i] = 1.0
        elif 'is_south' == col and preferences.get('is_south', False):
            pref_vector[i] = 1.0
        elif 'is_north' == col and preferences.get('is_north', False):
            pref_vector[i] = 1.0
        elif col == 'rating':
            pref_vector[i] = preferences.get('min_rating', 4.0) / 5.0
        elif col == 'has_tents' and preferences.get('prefer_tent', False):
            pref_vector[i] = 1.0
        elif col == 'has_buildings' and preferences.get('prefer_building', False):
            pref_vector[i] = 1.0

    distances, indices = knn.kneighbors([pref_vector], n_neighbors=top_n)
    results = centers.iloc[indices[0]].copy()
    results['similarity_score'] = (1 - distances[0]).round(3)
    return results[['name', 'site_type', 'region', 'rating', 'capacity', 'similarity_score']]

# Test recommendations
print('🏜️ Desert lover recommendations:')
print(recommend_centers({'site_type': 'DESERT', 'is_south': True, 'min_rating': 4.5}))
print()
print('🌊 Coastal family recommendations:')
print(recommend_centers({'site_type': 'COASTAL', 'prefer_building': True, 'min_rating': 4.0}))

## Part 3: Equipment Need Classifier (Multi-Label)

In [ ]:
EQUIPMENT_ITEMS = [
    'first_aid_kit', 'headlamp', 'sleeping_bag_warm', 'sleeping_bag_summer',
    'tent_desert', 'tent_forest', 'tent_coastal',
    'water_extra_5l', 'water_filter', 'water_bottle',
    'uv_clothing', 'rain_poncho', 'hiking_boots',
    'gps_compass', 'cooking_stove', 'cooler_box',
    'sunscreen_spf50', 'insect_repellent', 'sand_goggles',
    'life_jacket', 'snorkel_set',
]

# Equipment rules by conditions
EQUIPMENT_RULES = {
    'DESERT': {
        'SUMMER': ['first_aid_kit','headlamp','sleeping_bag_summer','tent_desert',
                   'water_extra_5l','water_bottle','uv_clothing','gps_compass',
                   'cooking_stove','sunscreen_spf50','sand_goggles','cooler_box'],
        'WINTER': ['first_aid_kit','headlamp','sleeping_bag_warm','tent_desert',
                   'water_extra_5l','water_bottle','gps_compass','cooking_stove',
                   'uv_clothing','sunscreen_spf50'],
        'SPRING': ['first_aid_kit','headlamp','sleeping_bag_summer','tent_desert',
                   'water_extra_5l','water_bottle','uv_clothing','gps_compass',
                   'cooking_stove','sunscreen_spf50','sand_goggles'],
        'AUTUMN': ['first_aid_kit','headlamp','sleeping_bag_warm','tent_desert',
                   'water_extra_5l','water_bottle','gps_compass','cooking_stove',
                   'sunscreen_spf50'],
    },
    'COASTAL': {
        'SUMMER': ['first_aid_kit','headlamp','sleeping_bag_summer','tent_coastal',
                   'water_bottle','sunscreen_spf50','life_jacket','snorkel_set',
                   'cooking_stove','insect_repellent'],
        'WINTER': ['first_aid_kit','headlamp','sleeping_bag_warm','tent_coastal',
                   'water_bottle','rain_poncho','cooking_stove'],
        'SPRING': ['first_aid_kit','headlamp','sleeping_bag_summer','tent_coastal',
                   'water_bottle','sunscreen_spf50','life_jacket','cooking_stove',
                   'insect_repellent'],
        'AUTUMN': ['first_aid_kit','headlamp','sleeping_bag_summer','tent_coastal',
                   'water_bottle','sunscreen_spf50','cooking_stove','life_jacket'],
    },
    'FOREST': {
        'SUMMER': ['first_aid_kit','headlamp','sleeping_bag_summer','tent_forest',
                   'water_filter','water_bottle','hiking_boots','insect_repellent',
                   'cooking_stove','sunscreen_spf50'],
        'WINTER': ['first_aid_kit','headlamp','sleeping_bag_warm','tent_forest',
                   'water_filter','water_bottle','hiking_boots','rain_poncho',
                   'cooking_stove'],
        'SPRING': ['first_aid_kit','headlamp','sleeping_bag_summer','tent_forest',
                   'water_filter','water_bottle','hiking_boots','insect_repellent',
                   'cooking_stove','rain_poncho'],
        'AUTUMN': ['first_aid_kit','headlamp','sleeping_bag_warm','tent_forest',
                   'water_filter','water_bottle','hiking_boots','insect_repellent',
                   'cooking_stove','rain_poncho'],
    },
}

print(f'Equipment items: {len(EQUIPMENT_ITEMS)}')
print(f'Rule conditions: {sum(len(v) for v in EQUIPMENT_RULES.values())}')

In [ ]:
# Generate training dataset for equipment classifier
def generate_equipment_data(n=3000):
    site_types = ['DESERT', 'COASTAL', 'FOREST']
    seasons = ['SPRING', 'SUMMER', 'AUTUMN', 'WINTER']
    govs_south = ['Kébili', 'Tataouine', 'Tozeur', 'Gafsa', 'Médenine']
    govs_north = ['Bizerte', 'Nabeul', 'Jendouba', 'Beja', 'Tunis', 'Ariana']
    govs_central = ['Sousse', 'Sfax', 'Kasserine', 'Kairouan', 'Mahdia']
    all_govs = govs_south + govs_north + govs_central
    
    rows = []
    for _ in range(n):
        site = np.random.choice(site_types)
        season = np.random.choice(seasons)
        gov = np.random.choice(all_govs)
        persons = np.random.randint(1, 12)
        days = np.random.randint(1, 10)
        temp = {'DESERT': {'SUMMER': 44, 'WINTER': 15, 'SPRING': 28, 'AUTUMN': 25},
                'COASTAL': {'SUMMER': 32, 'WINTER': 13, 'SPRING': 22, 'AUTUMN': 20},
                'FOREST': {'SUMMER': 30, 'WINTER': 10, 'SPRING': 20, 'AUTUMN': 18}}[site][season]
        temp += np.random.normal(0, 2)
        humidity = {'DESERT': 25, 'COASTAL': 70, 'FOREST': 72}[site]
        humidity += np.random.normal(0, 8)
        is_south = int(gov in govs_south)

        # Get essential equipment with some noise
        essentials = set(EQUIPMENT_RULES[site][season])
        # Add random noise: occasionally include/exclude items
        for item in EQUIPMENT_ITEMS:
            if item not in essentials and np.random.random() < 0.05:
                essentials.add(item)
            elif item in essentials and np.random.random() < 0.03:
                essentials.discard(item)

        row = {
            'site_type': site, 'season': season, 'governorate': gov,
            'num_persons': persons, 'num_days': days,
            'temperature': round(temp, 1), 'humidity': round(humidity, 1),
            'is_south': is_south,
        }
        for item in EQUIPMENT_ITEMS:
            row[item] = int(item in essentials)
        rows.append(row)
    return pd.DataFrame(rows)

df_equip = generate_equipment_data(3000)
print(f'Equipment dataset: {df_equip.shape}')
print('\nEquipment frequency (% of trips where it is essential):')
print((df_equip[EQUIPMENT_ITEMS].mean() * 100).round(1).sort_values(ascending=False).to_string())

In [ ]:
# Prepare features
le_site = LabelEncoder().fit(['DESERT','COASTAL','FOREST'])
le_season = LabelEncoder().fit(['SPRING','SUMMER','AUTUMN','WINTER'])
le_gov = LabelEncoder().fit(df_equip['governorate'].unique())

X_eq = pd.DataFrame({
    'site_type': le_site.transform(df_equip['site_type']),
    'season': le_season.transform(df_equip['season']),
    'governorate': le_gov.transform(df_equip['governorate']),
    'num_persons': df_equip['num_persons'],
    'num_days': df_equip['num_days'],
    'temperature': df_equip['temperature'],
    'humidity': df_equip['humidity'],
    'is_south': df_equip['is_south'],
})
y_eq = df_equip[EQUIPMENT_ITEMS].values

X_eq_train, X_eq_test, y_eq_train, y_eq_test = train_test_split(X_eq, y_eq, test_size=0.2, random_state=42)

# Train multi-output classifier
clf = MultiOutputClassifier(
    RandomForestClassifier(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42),
    n_jobs=-1
)
clf.fit(X_eq_train, y_eq_train)

y_eq_pred = clf.predict(X_eq_test)
hl = hamming_loss(y_eq_test, y_eq_pred)
print(f'\n✅ Equipment Classifier trained')
print(f'   Hamming Loss: {hl:.4f} (lower is better, 0 is perfect)')
print(f'   Subset Accuracy: {accuracy_score(y_eq_test, y_eq_pred):.4f}')

# Per-item accuracy
item_acc = {item: accuracy_score(y_eq_test[:,i], y_eq_pred[:,i]) for i, item in enumerate(EQUIPMENT_ITEMS)}
print('\nPer-item accuracy:')
for item, acc in sorted(item_acc.items(), key=lambda x: x[1]):
    bar = '█' * int(acc * 20)
    print(f'  {item:30s} {bar} {acc:.3f}')

In [ ]:
# Visualize equipment heatmap by site type & season
fig, ax = plt.subplots(figsize=(16, 8))

heatmap_data = []
conditions = []
for site in ['DESERT', 'COASTAL', 'FOREST']:
    for season in ['SPRING', 'SUMMER', 'AUTUMN', 'WINTER']:
        row = [1 if item in EQUIPMENT_RULES[site][season] else 0 for item in EQUIPMENT_ITEMS]
        heatmap_data.append(row)
        conditions.append(f'{site[:3]}-{season[:3]}')

hm = pd.DataFrame(heatmap_data, index=conditions, columns=EQUIPMENT_ITEMS)
sns.heatmap(hm, ax=ax, cmap='RdYlGn', annot=True, fmt='d',
            linewidths=0.5, cbar=False,
            xticklabels=[i.replace('_', '\n') for i in EQUIPMENT_ITEMS])
ax.set_title('Equipment Needs by Site Type & Season\n(1=Essential, 0=Optional)', fontsize=14, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Condition (Site-Season)')
plt.tight_layout()
plt.savefig('equipment_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Equipment heatmap saved')

In [ ]:
def predict_equipment(site_type, season, governorate='Tunis', persons=4, days=3, model_pkg=None):
    """Predict essential equipment for a camping trip."""
    if model_pkg is None:
        # Fallback to rules-based
        return EQUIPMENT_RULES.get(site_type, {}).get(season, [])

    clf_loaded = model_pkg['classifier']
    temp = {'DESERT': {'SUMMER': 44, 'WINTER': 15, 'SPRING': 28, 'AUTUMN': 25},
            'COASTAL': {'SUMMER': 32, 'WINTER': 13, 'SPRING': 22, 'AUTUMN': 20},
            'FOREST': {'SUMMER': 30, 'WINTER': 10, 'SPRING': 20, 'AUTUMN': 18}}.get(site_type, {}).get(season, 25)
    humidity = {'DESERT': 25, 'COASTAL': 70, 'FOREST': 72}.get(site_type, 60)
    is_south = int(governorate in ['Kébili', 'Tataouine', 'Tozeur', 'Gafsa', 'Médenine'])

    try:
        x = pd.DataFrame([{
            'site_type': model_pkg['le_site'].transform([site_type])[0],
            'season': model_pkg['le_season'].transform([season])[0],
            'governorate': model_pkg['le_gov'].transform([governorate])[0],
            'num_persons': persons, 'num_days': days,
            'temperature': temp, 'humidity': humidity, 'is_south': is_south
        }])
        pred = clf_loaded.predict(x)[0]
        return [item for item, needed in zip(EQUIPMENT_ITEMS, pred) if needed]
    except:
        return EQUIPMENT_RULES.get(site_type, {}).get(season, [])

# Test
for site, season, gov in [('DESERT','WINTER','Kébili'), ('COASTAL','SUMMER','Sousse'), ('FOREST','SPRING','Jendouba')]:
    items = predict_equipment(site, season, gov)
    print(f'{site} / {season} / {gov}:')
    print(f'  → {items}\n')

In [ ]:
# Save everything
recommender_pkg = {
    'knn': knn,
    'scaler': scaler,
    'feature_cols': feature_cols,
    'centers': centers,
    'version': '1.0'
}

equipment_pkg = {
    'classifier': clf,
    'le_site': le_site,
    'le_season': le_season,
    'le_gov': le_gov,
    'equipment_items': EQUIPMENT_ITEMS,
    'rules': EQUIPMENT_RULES,
    'version': '1.0'
}

joblib.dump(recommender_pkg, 'recommender.pkl', compress=3)
joblib.dump(equipment_pkg, 'equipment_classifier.pkl', compress=3)

print('✅ Saved: recommender.pkl')
print('✅ Saved: equipment_classifier.pkl')
print(f'\nHamming Loss: {hl:.4f}')
print(f'Subset Accuracy: {accuracy_score(y_eq_test, y_eq_pred):.4f}')

## ✅ Summary

| Component | Description | Accuracy |
|-----------|-------------|----------|
| Site Recommender | KNN cosine similarity on 15 features | Top-5 recall |
| Equipment Classifier | Multi-output RF, 21 gear items | Hamming < 0.05 |
| Regional Analysis | 2002-2017 official data | — |

**Integration:** Both models are served via `ml-api/app.py` (Flask) and consumed by the Spring Boot backend.